# Enriching titles with TMDB

Fills the gaps in the processed `movies` and `tv_shows` tables (made by `clean.ipynb`) with data from the [TMDB API](https://developer.themoviedb.org/docs/finding-data), and adds a few TMDB-only columns.

TMDB offers three ways to find a title:

- `/search`: text search by title. The Netflix data has only titles and years, so **this is the one used here**.
- `/find`: lookup by an external ID such as an IMDb ID. Netflix rows have no external IDs, so it is not needed.
- `/discover`: lists titles by filters. Not used, because it would add titles that are not on Netflix.

Steps:

1. Settings
2. Load the processed tables
3. TMDB request helper
4. Match a Netflix title to a TMDB ID (`/search`)
5. Fetch details and convert them to our columns
6. Cache results on disk
7. Fill the gaps
8. Check the result
9. Save and load

The API key is read from `.env` (`TMDB_API_KEY`, see `.env.example`).

In [1]:
# Importing libraries
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

## 1. Settings

- `SAMPLE` limits how many rows are processed. Set it to a small number (e.g. `20`) for a quick test run, or to `None` to process every row.
- `ONLY_ROWS_WITH_GAPS = True` looks up only rows with an empty Netflix column (about 3,500 instead of 8,800, roughly 2.5 times faster). The TMDB-only columns (budget, ratings, ...) are then filled only for those rows.
- Requests run in parallel (`MAX_WORKERS`). TMDB allows about 50 requests per second; rate-limited requests are retried automatically.

In [16]:
# override=True: .env always wins over a value already set in the kernel's environment
load_dotenv(find_dotenv(usecwd=True), override=True)
TMDB_API_KEY = os.getenv("TMDB_API_KEY")
assert TMDB_API_KEY and not TMDB_API_KEY.startswith("your_"), "Put your real TMDB_API_KEY in .env"

BASE_URL = "https://api.themoviedb.org/3"
PROCESSED_DIR = "../data/processed"
API_DIR = "../data/api"
CACHE_PATH = "../data/api/tmdb_cache.json"

SAMPLE = None                # e.g. 20 for a quick test run, None for all rows
ONLY_ROWS_WITH_GAPS = False  # True: skip rows whose Netflix columns are already complete
MAX_WORKERS = 32             # parallel requests
CAST_LIMIT = 10              # how many actors to keep per title

# Netflix columns that TMDB can fill when they are empty
FILL_COLUMNS = ["director", "cast", "country", "release_year", "rating", "duration_value"]

# New columns that come only from TMDB (budget and revenue exist only for movies)
TMDB_COLUMNS = {
    "movie": ["tmdb_id", "imdb_id", "genres", "tmdb_rating", "tmdb_votes", "budget", "revenue"],
    "tv": ["tmdb_id", "imdb_id", "genres", "tmdb_rating", "tmdb_votes"],
}

# TMDB country names that differ from the Netflix data
COUNTRY_NAMES = {
    "United States of America": "United States",
    "Russian Federation": "Russia",
}

## 2. Load the processed tables

The table shows how many values are missing in each column before enrichment. `date_added` is Netflix-specific, so TMDB cannot fill it.

In [3]:
df_movies = pd.read_csv(f"{PROCESSED_DIR}/movies.csv")
df_tv = pd.read_csv(f"{PROCESSED_DIR}/tv_shows.csv")

missing_before = pd.DataFrame({"movies": df_movies.isna().sum(), "tv_shows": df_tv.isna().sum()})
missing_before[missing_before.sum(axis=1) > 0]

,movies,tv_shows
director,188,2446
cast,475,350
country,440,391
date_added,0,10
rating,5,2


## 3. TMDB request helper

One shared session for all requests. Its connection pool is as large as `MAX_WORKERS`, so parallel requests reuse open connections instead of reconnecting each time. The key in `.env` can be either a v3 API key or a v4 read access token (the long one starting with `eyJ`). Requests that hit the rate limit (`429`) or a server error are retried with a growing pause, and a `404` returns an empty result.

In [4]:
session = requests.Session()
session.mount("https://", requests.adapters.HTTPAdapter(pool_connections=1, pool_maxsize=MAX_WORKERS))
if TMDB_API_KEY.startswith("eyJ"):  # v4 read access token
    session.headers["Authorization"] = f"Bearer {TMDB_API_KEY}"
    AUTH_PARAMS = {}
else:                               # v3 API key
    AUTH_PARAMS = {"api_key": TMDB_API_KEY}


def tmdb_get(path, **params):
    """GET a TMDB endpoint and return the JSON response."""
    params = {**AUTH_PARAMS, "language": "en-US", **params}
    for attempt in range(5):
        resp = session.get(f"{BASE_URL}{path}", params=params, timeout=15)
        if resp.status_code == 404:
            return {}
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(float(resp.headers.get("Retry-After", 2 ** attempt)))
            continue
        resp.raise_for_status()
        return resp.json()
    raise RuntimeError(f"TMDB request failed after 5 attempts: {path}")

## 4. Match a Netflix title to a TMDB ID

`/search` returns several candidates, so the best one is chosen:

1. The title (or original title) must match after ignoring case and punctuation.
2. Among those, the closest year wins, then the most popular result.
3. If no title matches exactly, the top result is accepted only if its year is within 1 of ours.

Movies are searched with the year first, which is more precise. If that finds nothing, the search is repeated without the year.

TV shows are searched without the year right away, because Netflix's `release_year` is usually the year of the latest season, not the first air date. A search with the year would often fail and cost an extra request. The year is still used to pick the closest result.

In [5]:
def normalize_title(text):
    return "".join(ch for ch in str(text).lower() if ch.isalnum())


def search_title(media, title, year=None):
    """/search/movie or /search/tv: text search, optionally narrowed by year."""
    params = {"query": title}
    if year:
        params["year" if media == "movie" else "first_air_date_year"] = year
    return tmdb_get(f"/search/{media}", **params).get("results", [])


def best_match(results, title, year, media):
    """Pick the result that matches the title and is closest in year."""
    if not results:
        return None
    date_field = "release_date" if media == "movie" else "first_air_date"
    target = normalize_title(title)

    def year_gap(r):
        date = r.get(date_field)
        return abs(int(date[:4]) - year) if (date and year) else 99

    def title_matches(r):
        names = [r.get("title"), r.get("original_title"), r.get("name"), r.get("original_name")]
        return target in {normalize_title(n) for n in names if n}

    exact = [r for r in results if title_matches(r)]
    if exact:
        return min(exact, key=lambda r: (year_gap(r), -r.get("popularity", 0)))
    return results[0] if year_gap(results[0]) <= 1 else None


def find_tmdb_id(media, title, year):
    search_year = year if media == "movie" else None
    match = best_match(search_title(media, title, search_year), title, year, media)
    if match is None and search_year:
        match = best_match(search_title(media, title), title, year, media)
    return match["id"] if match else None


find_tmdb_id("movie", "Dick Johnson Is Dead", 2020), find_tmdb_id("tv", "Blood & Water", 2021)

(653574, 103411)

## 5. Fetch details and convert them to our columns

One details request per title. `append_to_response` adds the cast and crew, the external IDs (for `imdb_id`) and the US age ratings, which use the same codes as Netflix (`PG-13`, `TV-MA`, ...).

How TMDB fields map to our columns:

| Column | Movies | TV shows |
|---|---|---|
| `director` | crew with job *Director* | show creators (`created_by`) |
| `cast` | first `CAST_LIMIT` actors | first `CAST_LIMIT` actors |
| `country` | production countries | production countries |
| `release_year` | release date | first air date |
| `rating` | US certification | US content rating |
| `duration_value` | runtime in minutes | number of seasons |

Note: Netflix rarely lists a director for TV shows, so for TV the show creators are used instead.

In [6]:
def join_names(names):
    names = [n for n in names if n]
    return ", ".join(names) or None


def get_details(media, tmdb_id):
    extra = "credits,external_ids," + ("release_dates" if media == "movie" else "content_ratings")
    return tmdb_get(f"/{media}/{tmdb_id}", append_to_response=extra)


def us_rating(details, media):
    """US age rating, e.g. PG-13 or TV-MA."""
    if media == "movie":
        for country in details.get("release_dates", {}).get("results", []):
            if country["iso_3166_1"] == "US":
                ratings = [r["certification"] for r in country["release_dates"] if r.get("certification")]
                return ratings[0] if ratings else None
    else:
        for country in details.get("content_ratings", {}).get("results", []):
            if country["iso_3166_1"] == "US":
                return country.get("rating") or None
    return None


def to_row(media, d):
    """Convert a TMDB details response into our column names."""
    credits = d.get("credits", {})
    if media == "movie":
        director = join_names(p["name"] for p in credits.get("crew", []) if p.get("job") == "Director")
        date, duration = d.get("release_date"), d.get("runtime")
    else:
        director = join_names(p["name"] for p in d.get("created_by", []))
        date, duration = d.get("first_air_date"), d.get("number_of_seasons")

    return {
        "director": director,
        "cast": join_names(p["name"] for p in credits.get("cast", [])[:CAST_LIMIT]),
        "country": join_names(COUNTRY_NAMES.get(c["name"], c["name"]) for c in d.get("production_countries", [])),
        "release_year": int(date[:4]) if date else None,
        "rating": us_rating(d, media),
        "duration_value": duration or None,
        "tmdb_id": d["id"],
        "imdb_id": d.get("external_ids", {}).get("imdb_id") or None,
        "genres": join_names(g["name"] for g in d.get("genres", [])),
        "tmdb_rating": d.get("vote_average") or None,
        "tmdb_votes": d.get("vote_count") or None,
        "budget": d.get("budget") or None,    # TMDB uses 0 for unknown
        "revenue": d.get("revenue") or None,
    }


to_row("movie", get_details("movie", find_tmdb_id("movie", "Dick Johnson Is Dead", 2020)))

{'director': 'Kirsten Johnson',
 'cast': 'Richard Johnson, Kirsten Johnson, Isla Sierck, Jed Sierck, Felix Torres, Viva Torres, Raymond Damazo, Michael Hilow, Michael Melton, Joanne Reiner Tucker',
 'country': 'United States',
 'release_year': 2020,
 'rating': 'PG-13',
 'duration_value': 89,
 'tmdb_id': 653574,
 'imdb_id': 'tt11394180',
 'genres': 'Documentary',
 'tmdb_rating': 7.1,
 'tmdb_votes': 149,
 'budget': None,
 'revenue': None}

## 6. Cache results on disk

Each lookup is saved to `data/api/tmdb_cache.json` under the key `title|year|media`, the same key format as `match_cache.json`. When the notebook runs again, cached titles make no requests. Titles that could not be matched are cached as `null`. To retry them, delete the cache file.

In [7]:
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, encoding="utf-8") as f:
        cache = json.load(f)
else:
    cache = {}


def save_cache():
    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache.copy(), f, ensure_ascii=False)


def lookup(media, title, year, tmdb_id=None):
    """Return the TMDB row for a title (from cache if possible), or None if no match."""
    key = f"{title}|{year}|{media}"
    if key not in cache:
        tmdb_id = tmdb_id or find_tmdb_id(media, title, year)
        details = get_details(media, tmdb_id) if tmdb_id else {}
        cache[key] = to_row(media, details) if details else None
    return cache[key]


len(cache)

500

## 7. Fill the gaps

For each table:

1. Add the TMDB columns if they do not exist yet.
2. Keep only rows that have at least one empty column we can fill (only empty Netflix columns if `ONLY_ROWS_WITH_GAPS`).
3. Look these rows up in parallel, saving the cache every 250 titles.
4. Write TMDB values **only into empty cells**. Existing Netflix values are never overwritten.

Failed requests are counted and not cached, so running the cell again retries them.

In [8]:
def is_missing(value):
    return value is None or (isinstance(value, str) and not value.strip()) or pd.isna(value)


def fill_table(df, media):
    df = df.copy()
    columns = FILL_COLUMNS + TMDB_COLUMNS[media]
    for col in columns:
        if col not in df.columns:
            df[col] = None
    df[columns] = df[columns].astype("object")

    check = FILL_COLUMNS if ONLY_ROWS_WITH_GAPS else columns
    todo = df[df[check].map(is_missing).any(axis=1)]
    if SAMPLE:
        todo = todo.head(SAMPLE)
    print(f"{media}: {len(todo)} of {len(df)} rows to look up")

    results, errors = {}, 0
    with ThreadPoolExecutor(MAX_WORKERS) as pool:
        futures = {
            pool.submit(
                lookup, media, row["title"],
                None if pd.isna(row["release_year"]) else int(row["release_year"]),
                None if pd.isna(row["tmdb_id"]) else int(row["tmdb_id"]),
            ): idx
            for idx, row in todo.iterrows()
        }
        for n, future in enumerate(as_completed(futures), 1):
            try:
                results[futures[future]] = future.result()
            except Exception as e:
                errors += 1
                print(f"  error on {df.at[futures[future], 'title']!r}: {e}")
            if n % 250 == 0:
                save_cache()
                print(f"  {n}/{len(futures)} done")
    save_cache()

    filled = dict.fromkeys(columns, 0)
    for idx, row in results.items():
        if row is None:
            continue
        for col in columns:
            if is_missing(df.at[idx, col]) and not is_missing(row.get(col)):
                df.at[idx, col] = row[col]
                filled[col] += 1

    unmatched = sum(row is None for row in results.values())
    print(f"{media}: matched {len(results) - unmatched}, unmatched {unmatched}, errors {errors}")
    return df, pd.Series(filled, name=media)


df_movies, filled_movies = fill_table(df_movies, "movie")
df_tv, filled_tv = fill_table(df_tv, "tv")

movie: 6131 of 6131 rows to look up
  250/6131 done
  500/6131 done
  750/6131 done
  1000/6131 done
  1250/6131 done
  1500/6131 done
  1750/6131 done
  2000/6131 done
  2250/6131 done
  2500/6131 done
  2750/6131 done
  3000/6131 done
  3250/6131 done
  3500/6131 done
  3750/6131 done
  4000/6131 done
  4250/6131 done
  4500/6131 done
  4750/6131 done
  5000/6131 done
  5250/6131 done
  5500/6131 done
  5750/6131 done
  6000/6131 done
movie: matched 5962, unmatched 169, errors 0
tv: 2676 of 2676 rows to look up
  250/2676 done
  500/2676 done
  750/2676 done
  1000/2676 done
  1250/2676 done
  1500/2676 done
  1750/2676 done
  2000/2676 done
  2250/2676 done
  2500/2676 done
tv: matched 2466, unmatched 210, errors 0


## 8. Check the result

How many cells were filled per column, and what is still missing.

In [9]:
pd.DataFrame({"movies": filled_movies, "tv_shows": filled_tv}).fillna(0).astype(int)

,movies,tv_shows
budget,1581,0
cast,341,186
country,295,260
director,112,1261
duration_value,0,0
genres,5845,2381
imdb_id,5863,2320
rating,2,1
release_year,0,0
revenue,1727,0


In [10]:
missing_after = pd.DataFrame({"movies": df_movies.isna().sum(), "tv_shows": df_tv.isna().sum()})
missing_after[missing_after.sum(axis=1) > 0]

,movies,tv_shows
budget,4550,NaN
cast,134,164.0
country,145,131.0
date_added,0,10.0
director,76,1185.0
genres,286,295.0
imdb_id,268,356.0
rating,3,1.0
revenue,4404,NaN
tmdb_id,169,210.0


## 9. Save and load

Whole-number columns are converted to `Int64`, so they are saved as `2020`, not `2020.0`. The enriched tables replace `movies.csv` and `tv_shows.csv` in `data/processed/` and the matching PostgreSQL tables, using the same settings as `clean.ipynb`.

In [17]:
int_columns = ["release_year", "duration_value", "tmdb_id", "tmdb_votes", "budget", "revenue"]
for df in (df_movies, df_tv):
    for col in int_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col]).astype("Int64")
    df["tmdb_rating"] = pd.to_numeric(df["tmdb_rating"])

df_movies.to_csv(f"{API_DIR}/movies.csv", index=False)
df_tv.to_csv(f"{API_DIR}/tv_shows.csv", index=False)

url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", 5432)),
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)

df_movies.to_sql("movies", engine, schema="public", if_exists="replace", index=False)
df_tv.to_sql("tv_shows", engine, schema="public", if_exists="replace", index=False)

676